<a href="https://colab.research.google.com/github/aayushsinghiitgn/Pandas_Practice/blob/main/Pands_prac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import pandas as pd
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/My Drive/Colab Notebooks/GroupPPT")
AS_OF = pd.Timestamp("2026-10-09 23:59:59")
members = pd.read_csv(DATA_DIR / "members.csv", parse_dates=["join_date"])
renewals = pd.read_csv(DATA_DIR / "renewal_opportunities.csv", parse_dates=["expiry_date", "renewal_deadline"])
events = pd.read_csv(DATA_DIR / "member_events.csv", parse_dates=["event_time"])

# Date columns supplied for the exercises.
members["join_year"] = members["join_date"].dt.year
renewals["expiry_year"] = renewals["expiry_date"].dt.year
renewals["expiry_month"] = renewals["expiry_date"].dt.month
print(members.shape, renewals.shape, events.shape)

Mounted at /content/drive
(8580, 9) (11580, 9) (152057, 10)


In [ ]:
members.head()

,member_id,join_date,plan_code,annual_fee,acquisition_channel,city,status_2025_10_09,status_2026_10_09,join_year
0,MP000467,2023-09-22,PLUS_STANDARD,1499,STORE,Gurugram,PAID,PAID,2023
1,MP005549,2025-09-04,PLUS_PREMIUM,2499,WEB,Bengaluru,PAID,PAID,2025
2,MP000708,2023-09-13,PLUS_STANDARD,1499,STORE,Bengaluru,PAID,PAID,2023
3,MP006266,2025-10-04,PLUS_STANDARD,1499,APP,Delhi,PAID,PAID,2025
4,MP000081,2023-09-26,PLUS_STANDARD,1499,APP,Jaipur,PAID,PAID,2023


In [ ]:
renewals.head()

,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9


In [ ]:
events.head()

,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27
2,EV0000747,MP001531,RN000060,2024-07-03 13:20:00,BENEFIT_USED,WEB,SUCCESS,FREE_DELIVERY,0,55
3,EV0029706,MP002497,RN002203,2024-07-03 17:39:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,53
4,EV0031926,MP001821,RN002456,2024-07-03 17:47:00,BENEFIT_USED,APP,SUCCESS,PRIORITY_SUPPORT,0,13


In [ ]:
result_1 = members.groupby("acquisition_channel").agg(
    member_count=("member_id", "count"),
    average_annual_fee=("annual_fee", "mean"),
    earliest_join=("join_date", "min"),
    latest_join=("join_date", "max")
)
result_1

,member_count,average_annual_fee,earliest_join,latest_join
acquisition_channel,,,,
APP,4328,1805.839187,2023-09-01,2026-10-01
STORE,1655,1799.302115,2023-09-01,2026-10-01
WEB,2597,1794.725838,2023-09-01,2026-10-01


In [ ]:
result_2 = members.loc[members["status_2026_10_09"] == "PAID"].groupby("city").agg(
    paid_members=("member_id", "count"),
    average_annual_fee=("annual_fee", "mean"),
    channels_present=("acquisition_channel", "nunique")
)
result_2

,paid_members,average_annual_fee,channels_present
city,,,
Ahmedabad,725,1799.689655,3
Bengaluru,771,1821.957198,3
Delhi,754,1786.798408,3
Gurugram,755,1782.443709,3
Hyderabad,789,1800.647655,3
Jaipur,778,1819.051414,3
Mumbai,754,1772.209549,3
Pune,698,1794.128940,3


In [ ]:
# Filter source rows for members who transitioned from PAID to EXPIRED
q3_filtered = members.loc[
    (members["status_2025_10_09"] == "PAID") &
    (members["status_2026_10_09"] == "EXPIRED")
]

q3_grouped = q3_filtered.groupby("acquisition_channel").agg(
    expired_members=("member_id", "count"),
    average_annual_fee=("annual_fee", "mean")
)

# Use .loc on the aggregated result to filter channels with >= 300 members
result_3 = q3_grouped.loc[q3_grouped["expired_members"] >= 300]
result_3

,expired_members,average_annual_fee
acquisition_channel,,
APP,619,1794.638126
WEB,391,1826.365729


In [ ]:
q4_filtered = members.loc[
    (members["join_year"].isin([2025, 2026])) &
    (members["status_2026_10_09"] == "PAID")
]

result_4 = q4_filtered.groupby(["join_year", "acquisition_channel"]).agg(
    paid_members=("member_id", "count"),
    cities_present=("city", "nunique"),
    average_annual_fee=("annual_fee", "mean")
)
result_4

paid_members  cities_present  \
join_year acquisition_channel                                 
2025      APP                           681               8   
          STORE                         238               8   
          WEB                           409               8   
2026      APP                           924               8   
          STORE                         351               8   
          WEB                           541               8   

                               average_annual_fee  
join_year acquisition_channel                      
2025      APP                         1798.559471  
          STORE                       1822.529412  
          WEB                         1775.283619  
2026      APP                         1807.441558  
          STORE                       1795.296296  
          WEB                         1794.748614

In [ ]:
q5_filtered = members.loc[
    (members["status_2026_10_09"] == "PAID") &
    (members["acquisition_channel"].isin(["APP", "WEB"]))
]

q5_grouped = q5_filtered.groupby("city").agg(
    paid_members=("member_id", "count"),
    plans_present=("plan_code", "nunique"),
    average_annual_fee=("annual_fee", "mean")
)

result_5 = q5_grouped.loc[
    (q5_grouped["paid_members"] >= 500) & (q5_grouped["average_annual_fee"] > 1750),
    ["paid_members", "average_annual_fee"]
]
result_5

,paid_members,average_annual_fee
city,,
Ahmedabad,577,1805.759099
Bengaluru,625,1823.800000
Delhi,613,1784.481240
Gurugram,601,1783.525790
Hyderabad,629,1804.246423
Jaipur,646,1825.625387
Mumbai,604,1765.556291
Pune,572,1790.958042


In [ ]:
q7_filtered = renewals.loc[
    (renewals["expiry_month"] == 9) &
    (renewals["expiry_year"].isin([2025, 2026]))
]

result_7 = q7_filtered.groupby("expiry_year").agg(
    members_due=("renewal_id", "count"),
    distinct_members=("member_id", "nunique"),
    first_expiry=("expiry_date", "min"),
    last_expiry=("expiry_date", "max")
)
result_7

,members_due,distinct_members,first_expiry,last_expiry
expiry_year,,,,
2025,4000,4000,2025-09-01,2025-09-30
2026,4480,4480,2026-09-01,2026-09-30


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "City": ["Delhi", "Delhi", "Jaipur", "Jaipur", "Mumbai", "Mumbai"],
    "Product": ["Pen", "Notebook", "Pen", "Notebook", "Notebook", "Pen"],
    "Sales": [200, 500, 150, 350, 600, 400]
})
df

,City,Product,Sales
0,Delhi,Pen,200
1,Delhi,Notebook,500
2,Jaipur,Pen,150
3,Jaipur,Notebook,350
4,Mumbai,Notebook,600
5,Mumbai,Pen,400


In [ ]:
def failure_percentage(ds):
    return ds.unique/ds.min()

In [ ]:
events.loc[events['event_type'] == 'PAYMENT_ATTEMPT'].groupby('channel').agg(event_count = ('event_id', 'count'),
                                 )

,event_count
channel,
APP,7445
WEB,5863


In [ ]:
renewals.head()

,renewal_id,member_id,renewal_number,expiry_date,renewal_deadline,renewal_fee,offered_discount,expiry_year,expiry_month
0,RN004822,MP002964,1,2025-09-02,2025-09-09,2499,0,2025,9
1,RN007937,MP001571,3,2026-09-14,2026-09-21,2499,0,2026,9
2,RN002399,MP002040,1,2024-09-05,2024-09-12,1499,0,2024,9
3,RN009951,MP004511,1,2026-09-18,2026-09-25,1499,0,2026,9
4,RN006488,MP003536,1,2025-09-13,2025-09-20,1499,0,2025,9


In [ ]:
def average_high_cost(ds):
  cost_mean = ds[ds > 30]
  if ds.empty:
    return 0
  else:
   return cost_mean.mean()

In [ ]:
events.groupby('event_type').agg(total_events = ('event_id','count'),
                                     average_high_cost = ('event_cost', average_high_cost))

,total_events,average_high_cost
event_type,,
BENEFIT_USED,125995,48.035811
PAYMENT_ATTEMPT,13308,NaN
REMINDER_SENT,11580,NaN
SUPPORT_CONTACT,1174,45.000000


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Department": ["Sales", "Sales", "Sales", "Support", "Support", "Support"],
    "Salary": [40000, 55000, 65000, 30000, 45000, 50000],
    "Experience": [1, 3, 5, 2, 4, 1]
})


# Question: For each department, find the average salary of employees with at least three years of experience.

In [ ]:

# Using member_events.csv, select only "PAYMENT_ATTEMPT" events.
# For each channel, calculate the average payment amount of successful attempts only.

# Write a function named successful_payment_mean(group) that:

# Receives the whole group DataFrame.
# Selects rows where outcome == "SUCCESS".
# Returns the mean of amount for those rows.
# Use groupby().apply() and name the result column average_successful_payment when resetting the index.
# Failed attempts must not contribute to the average.

In [ ]:
events.head(2)

,event_id,member_id,renewal_id,event_time,event_type,channel,outcome,detail_code,amount,event_cost
0,EV0023859,MP000456,RN001802,2024-07-03 08:50:00,BENEFIT_USED,APP,SUCCESS,MEMBER_OFFER,0,10
1,EV0005255,MP000612,RN000390,2024-07-03 11:34:00,BENEFIT_USED,APP,SUCCESS,FREE_DELIVERY,0,27


In [ ]:
def successful_payment_mean(dff):
  successful_payment_mean = dff[dff['outcome'] == 'SUCCESS']['amount'].mean()
  return successful_payment_mean


In [ ]:
events.groupby('channel').apply(successful_payment_mean).reset_index(name='average_successful_payment')

/tmp/ipykernel_2218/3333402009.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  events.groupby('channel').apply(successful_payment_mean).reset_index(name='average_successful_payment')


,channel,average_successful_payment
0,APP,118.950354
1,EMAIL,NaN
2,SUPPORT,NaN
3,WEB,118.752943


In [ ]:
# Finance wants to know which payment channel costs Kalpa more for each successful renewal.
# Using member_events.csv, calculate the payment-processing cost per successful renewal for each channel. Include the costs of failed attempts as well—they still cost Kalpa money.
# Which channel appears more expensive, and why might comparing only its average cost per attempt be misleading?

In [ ]:
payments = events[events['event_type'] == 'PAYMENT_ATTEMPT']
payments['event_cost'].sum()

np.int64(116856)

In [ ]:
(payments['outcome'] == 'SUCCESS').sum()

np.int64(9024)

In [ ]:
# dff = payments[payments['channel'] == 'APP']
import numpy as np
def get_cost_per_success(dff):
  total_cost = dff['event_cost'].sum()
  success = dff[dff['outcome'] == 'SUCCESS']['event_cost'].sum()
  if(success>0):
    return total_cost/success
  return np.nan

In [ ]:
payments.groupby('channel').apply(get_cost_per_success).reset_index(name = 'cost_per_success')

/tmp/ipykernel_2218/1561518105.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  payments.groupby('channel').apply(get_cost_per_success).reset_index(name = 'cost_per_success')


,channel,cost_per_success
0,APP,1.107733
1,WEB,1.050385


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "City": ["Delhi", "Jaipur", "Delhi", "Jaipur"],
    "Sales": [100, 200, 300, 600]
})

df

,City,Sales
0,Delhi,100
1,Jaipur,200
2,Delhi,300
3,Jaipur,600


In [ ]:
df['Sales']/df['Sales'].sum()

,Sales
0,0.083333
1,0.166667
2,0.250000
3,0.500000


In [ ]:
def contri(ds):
  return ds['Sales']/ds['Sales'].sum()

In [ ]:
df.groupby('City').apply(contri).reset_index()

/tmp/ipykernel_2218/3865941527.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('City').apply(contri).reset_index()


,City,level_1,Sales
0,Delhi,0,0.25
1,Delhi,2,0.75
2,Jaipur,1,0.25
3,Jaipur,3,0.75


In [ ]:
# 1. Identify unusually expensive events
# Within each event_type, calculate its average event_cost.
# Return a Boolean Series indicating whether each event costs more than its group’s average. Assign it to:
# events["above_average_cost"]

In [ ]:
dff = events.groupby(events['event_type']).agg(event_type = ('event_type','count'), average_cost = ('event_cost','mean'))
dff

,event_type,average_cost
event_type,,
BENEFIT_USED,125995,35.925584
PAYMENT_ATTEMPT,13308,8.780884
REMINDER_SENT,11580,1.000000
SUPPORT_CONTACT,1174,45.000000


In [ ]:
# 2. Find each event’s share of channel cost
# Within each channel, calculate what percentage of the channel’s total event cost each row contributes.
# Return a Series and assign it to:
# events["channel_cost_share_pct"]

In [ ]:
def channel_cost(dff):
  return dff['event_cost']/dff['event_cost'].sum()*100

In [ ]:
events.groupby('channel').apply(channel_cost).reset_index(drop = True)

/tmp/ipykernel_2218/2913880891.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  events.groupby('channel').apply(channel_cost).reset_index(drop = True)


,event_cost
0,0.000429
1,0.001159
2,0.002276
3,0.000558
4,0.002061
...,...
152052,0.000519
152053,0.000086
152054,0.000086
152055,0.000086
